In [5]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [6]:
from rag_helper import RAGBase
from ingest import build_index,load_faq_data


In [7]:
documents = load_faq_data()
index=build_index(documents)

In [8]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

In [9]:
# asking without tool
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—usually you can still join, but it depends on the course’s enrollment status and start date.\n\nA few things to check:\n- **Enrollment deadline:** Some courses allow late enrollment, others don’t.\n- **Seat availability:** If it’s full, you may need to join a waitlist.\n- **Prerequisites/eligibility:** Some courses require prior courses or approval.\n- **Access to past content:** If the course already started, you may be able to catch up, but you might miss earlier assignments or live sessions.\n\nIf you want, I can help you figure out the exact next step if you tell me:\n1. the course name,  \n2. the platform or school, and  \n3. whether it has already started.'

In [10]:
# Defining the tool for the agent to use
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [11]:
# search_tool is a dictionary that defines the structure and parameters of the search function,
# which can be used by an agent to perform searches in the FAQ database. 
# It specifies the type of tool, its name, description, and the expected parameters for the search query.

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [12]:
# sending the question with search tool to the model
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course enrollment discovered course can I join"}', call_id='call_hn9NSlr9RhbKaBGK1CT1U0Sg', name='search', type='function_call', id='fc_00033ef6bd7b4d41006a4134cd9cfc81a29e4fcbbc64ea93e5', namespace=None, status='completed')]

In [13]:
# Executing the function and sending the result back
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [14]:
# sending the result back to the model
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [15]:
# Asking the model again
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [16]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(782, 31)

In [17]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(770, 34)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001359


In [18]:
# Agentic loop
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [19]:
# a function call helper
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [20]:
# processing the question with the agentic loop

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course enrollment discovered the course can I join"}
function_call: search {"query":"course registration late enrollment can I join"}
function_call: search {"query":"access course after start join course FAQ"}


In [21]:
# The function call loop continues until there are no more function calls to process.
 
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join.

If you want a certificate, make sure you submit your project while submissions are still open. You can also start learning and submitting homework even if you didn’t register, as long as the submission form is open.

If you'd like, I can also help you figure out the best way to start the course now. Are there other areas you want to explore?


In [22]:
# wrapping the agentic loop into a function for reusability

def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [23]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama local run install start local model server FAQ"}
function_call: search {"query":"run Ollama locally command ollama serve pull model FAQ"}
function_call: search {"query":"local Ollama setup macOS Windows Linux FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: install the `.pkg`
     - **Windows**: install the `.msi`
     - **Linux**:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This will download the model and start a local chat session.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from Ollama.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='ll

'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: install the `.pkg`\n     - **Windows**: install the `.msi`\n     - **Linux**:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and start a local chat session.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from Ollama.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection issue, you can restart the server with:\n```bash\nollama serve\n```\nor in a notebook:\n```bash\n!nohup oll

In [24]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late access FAQ"}
iteration #2...
function_call: search {"query":"confirmation email accepted start learning submitting homework while form open registration gauge interest no need to register FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

A couple of important notes:
- You can start learning anytime.
- If you want a certificate, you need to submit your project while submissions are still open.
- Registration is mainly to gauge interest, and you can usually start learning/submitting homework while the form is open.

If you want, I can also help you find the course start steps, homework links, or certificate rules.


'Yes — you can still join the course.\n\nA couple of important notes:\n- You can start learning anytime.\n- If you want a certificate, you need to submit your project while submissions are still open.\n- Registration is mainly to gauge interest, and you can usually start learning/submitting homework while the form is open.\n\nIf you want, I can also help you find the course start steps, homework links, or certificate rules.'

In [25]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen gambit definition"}
iteration #2...
function_call: search {"query":"\"queen gambit\" course FAQ chess opening"}
iteration #3...
ASSISTANT:
A “queen gambit” usually means the **Queen’s Gambit** in chess: an opening that starts with

1. **d4 d5**
2. **c4**

White offers the c-pawn to try to control the center and gain a positional advantage.

If you meant something else, like the TV show *The Queen’s Gambit*, I can explain that too.  
Want a simple chess-board explanation of why it works?


'A “queen gambit” usually means the **Queen’s Gambit** in chess: an opening that starts with\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the c-pawn to try to control the center and gain a positional advantage.\n\nIf you meant something else, like the TV show *The Queen’s Gambit*, I can explain that too.  \nWant a simple chess-board explanation of why it works?'

In [ ]:
# instructions updated to dont answer off-topic questions

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"Queen's Gambit chess opening course"}
iteration #3...
ASSISTANT:
It looks like “queen gambit” isn’t in the course FAQ, so I can’t answer it from the course materials. If you meant **Queen’s Gambit** as a chess term, that’s outside the course scope.

Is there another course-related topic or logistics question you’d like to explore?


'It looks like “queen gambit” isn’t in the course FAQ, so I can’t answer it from the course materials. If you meant **Queen’s Gambit** as a chess term, that’s outside the course scope.\n\nIs there another course-related topic or logistics question you’d like to explore?'

In [27]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [28]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [29]:
# letting the agent to use the search tool to answer questions about the course

def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [30]:
# register without passing schema

agent_tools = Tools()
agent_tools.add_tool(search)

In [31]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [32]:
# chat interface and runner setup for the agentic loop

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [37]:
# running the agentic loop with the runner and displaying the conversation in the chat interface

result = runner.loop(
    prompt="How do I run Ollama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [38]:
# trying with typing error

result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [34]:
# cost
result.cost

CostInfo(input_cost=Decimal('0.002688'), output_cost=Decimal('0.0014625'), total_cost=Decimal('0.0041505'))

In [35]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local Ollama course FAQ"}', call_id='call_7o

In [36]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [39]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='can i use another frameworks', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"use another frameworks frameworks allow